# Manuscript figures from the kit outputs
Figures 1 to 4 are drawn by this notebook. Figures 2, 3 and 4 read the kit's own output text files; Figure 1 is a diagram drawn in code from the cohort counts. Figures 5 and S1 are saved by notebooks 02 and 03 of the kit (`fig_clinical_coefficients.png`, `figS1_curves.png`).

In [ ]:
import os, re, ast
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass
REPORTS = '/content/drive/MyDrive/CVES Fresh/CVES Repair/reports'   # folder with 3_clinical.txt, 4_grid.txt, 7_robustness.txt
OUTDIR  = '/content/drive/MyDrive/CVES Fresh/CVES Repair/figures_code'
os.makedirs(OUTDIR, exist_ok=True)
OUT = lambda n: os.path.join(OUTDIR, n)
GRID, ROBUST, CLINICAL = (os.path.join(REPORTS, f) for f in ('4_grid.txt', '7_robustness.txt', '3_clinical.txt'))
plt.rcParams['font.family'] = 'DejaVu Sans'
DK, CO, TE, GO, GR, OR = '#264653', '#E76F51', '#2A9D8F', '#E9C46A', '#E9EEF2', '#FDF3E9'
def strip(ax):
    for s in ('top', 'right'): ax.spines[s].set_visible(False)

## Figure 1. Study design and cohort cascade

In [ ]:
# ---------- Figure 1: design ----------
fig,ax=plt.subplots(figsize=(7.2,8.6),dpi=250); ax.set_xlim(0,10); ax.set_ylim(0,12); ax.axis('off')
def box(x,y,w,h,txt,fc=GR,ec='#2F5D8A',fs=8.6,bold=False):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.02,rounding_size=0.15',fc=fc,ec=ec,lw=1.4))
    ax.text(x+w/2,y+h/2,txt,ha='center',va='center',fontsize=fs,fontweight='bold' if bold else 'normal',linespacing=1.35)
def arr(x1,y1,x2,y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle='-|>',mutation_scale=11,lw=1.2,color='#222'))
box(1.6,10.5,6.8,1.15,'CVES v1.0.0 (PhysioNet)\n172 participant records (84 stroke, 88 control)',bold=True)
arr(5,10.5,5,9.95)
box(2.2,8.8,5.6,1.15,'Completed visits: 91 (43 stroke, 48 control)\n4 investigator-excluded, 1 blank status')
arr(3.6,8.8,2.6,8.15); arr(6.4,8.8,7.4,8.15)
box(0.2,6.6,4.6,1.55,'Sit-stand-balance cohort\nn = 64 (37 stroke, 27 control)\nrecordings kept for 86% of stroke and\n56% of control completed visits')
box(5.2,6.6,4.6,1.55,'Head-up-tilt cohort\nn = 68 (40 stroke, 28 control)\n63 participants are in both cohorts')
arr(2.5,6.6,2.5,5.95); arr(7.5,6.6,7.5,5.95)
box(0.2,3.7,4.6,2.25,'Recording audit and phase track\nMCA regimes checked against the lab table\nand ABP-to-MCA timing; CARNet 2022\ntransfer function analysis; three\nalignment arms, full grid')
box(5.2,3.7,4.6,2.25,'Clinical track (n = 64; n = 91 check)\n15 variables: R (10 risk factors),\nP (gait speed + 4 medication classes),\nA = R + P; repeated 5-fold CV,\nleave-one-out, refit bootstrap (2000)')
arr(2.5,3.7,2.5,3.05); arr(7.5,3.7,7.5,3.05)
box(0.2,1.65,4.6,1.4,'Questions 1 and 3\nAre the MCA recordings consistent?\nIs low-frequency phase different?\n(Figures 2, 4; Tables 3, 7)',fc=OR,ec='#C0642B')
box(5.2,1.65,4.6,1.4,'Question 2\nWhat carries the classifiable signal,\nand do signals add to it?\n(Figures 3, 5; Tables 4 to 6)',fc=OR,ec='#C0642B')
arr(2.5,1.65,4.2,1.05); arr(7.5,1.65,5.8,1.05)
box(1.4,0.1,7.2,0.95,'Participant-level bootstrap intervals throughout;\nanalysis choices shown side by side, not selected')
fig.savefig(OUT('fig1_design.png'),bbox_inches='tight',pad_inches=0.08); plt.show()

## Figure 2. Recording regimes

In [ ]:
rob = open(ROBUST, encoding='utf8').read()
cli = open(CLINICAL, encoding='utf8').read()
PROTO = [('head-up-tilt', 'Supine (head-up tilt)'), ('sit-stand-balance', 'Seated (sit-stand-balance)')]
def clean(s): return re.sub(r'np\.float64\(([^)]*)\)', r'\1', s)
# ------------------------------------------------ Figure 2
share, shift, auc = {}, {}, {}
for key, _ in PROTO:
    m = re.search(re.escape(key) + r' all \[\[(\d+), (\d+)\], \[(\d+), (\d+)\]\]', rob)
    sh, sl, ch, cl = map(int, m.groups())
    share[key] = {'Stroke': (sh, sh + sl), 'Control': (ch, ch + cl)}
    for grp in ('stroke', 'control'):
        m = re.search(re.escape(key) + ' ' + grp + r' n lo/hi \d+ \d+ \| ECG-ABP (-?\d+) ms \| ECG-MCA -?\d+ ms \| ABP-MCA lag (-?\d+) ms', rob)
        shift[(key, grp)] = (int(m.group(1)), int(m.group(2)))
# order/regime/group AUCs appear in the line that follows each 'valid only' line, in file order
oa = re.findall(r'order->regime ([\d.]+); order->group ([\d.]+); regime->group ([\d.]+)', rob)
for (key, _), v in zip(PROTO, oa): auc[key] = tuple(map(float, v))

fig, (A, B, C) = plt.subplots(1, 3, figsize=(8.2, 3.4), dpi=250, gridspec_kw={'width_ratios': [1, 1.25, 1.1]})
x = 0; ticks = []; labs = []
for key, lab in PROTO:
    for g, col in (('Stroke', DK), ('Control', CO)):
        h, n = share[key][g]
        A.bar(x, 100 * h / n, color=col, width=0.8, label=g if key == PROTO[0][0] else None)
        A.text(x, 100 * h / n + 2, f'{h}/{n}', ha='center', fontsize=6.5)
        x += 1
    ticks.append(x - 1.5); labs.append(lab.split(' (')[0] + '\n(' + lab.split('(')[1]); x += 0.7
A.set_xticks(ticks); A.set_xticklabels(labs, fontsize=7); A.set_ylim(0, 100)
A.set_ylabel('Records in the high regime (%)', fontsize=8); A.tick_params(labelsize=7); A.legend(fontsize=7, frameon=False, loc='lower right', bbox_to_anchor=(1, 1), ncol=2, borderaxespad=0.2)
A.text(-0.02, 1.12, '(a)', transform=A.transAxes, fontweight='bold', fontsize=9); strip(A)

y = 0; yt = []; yl = []
for key, lab in PROTO:
    for grp in ('stroke', 'control'):
        ecg, lag = shift[(key, grp)]
        B.barh(y + 0.18, lag, height=0.34, color=TE, label='ABP to MCA' if y == 0 else None)
        B.barh(y - 0.18, ecg, height=0.34, color=GO, label='ECG to ABP' if y == 0 else None)
        yt.append(y); yl.append(f"{lab.split(' (')[0]}, {grp}"); y += 1
B.axvline(0, color='grey', lw=0.8); B.set_yticks(yt); B.set_yticklabels(yl, fontsize=7); B.invert_yaxis()
B.set_xlabel('High minus low regime, median (ms)', fontsize=8); B.tick_params(labelsize=7); B.legend(fontsize=7, frameon=False, loc='lower right', bbox_to_anchor=(1, 1), ncol=2, borderaxespad=0.2)
B.text(-0.02, 1.12, '(b)', transform=B.transAxes, fontweight='bold', fontsize=9); strip(B)

names = ['Order\nto regime', 'Order\nto group', 'Regime\nto group']
for i, (key, lab) in enumerate(PROTO):
    C.bar([j + (i - 0.5) * 0.38 for j in range(3)], auc[key], width=0.36, color=[DK, TE][i], label=lab.split(' (')[0])
C.axhline(0.5, color='grey', lw=0.8); C.set_ylim(0.4, 1.0); C.set_xticks(range(3)); C.set_xticklabels(names, fontsize=7)
C.set_ylabel('AUC', fontsize=8); C.tick_params(labelsize=7); C.legend(fontsize=7, frameon=False, loc='lower right', bbox_to_anchor=(1, 1), ncol=2, borderaxespad=0.2)
C.text(-0.02, 1.12, '(c)', transform=C.transAxes, fontweight='bold', fontsize=9); strip(C)
fig.tight_layout(); fig.savefig(OUT('fig2_regimes.png'), bbox_inches='tight', pad_inches=0.08); plt.show()

## Figure 3. Clinical variable sets and Shapley shares

In [ ]:
# ------------------------------------------------ Figure 3
pt = ast.literal_eval(re.search(r'point \(100 reps\): (\{.*?\})', cli).group(1))
ci = ast.literal_eval(clean(re.search(r'refit OOB CI: (\{.*\})', cli).group(1)))
a91 = ast.literal_eval(re.search(r'all completed visits n 91.*?(\{.*\})', cli).group(1))
sets = [('R', 'R: risk factors'), ('R+gait', 'R + gait speed'), ('R+meds', 'R + medications'), ('A', 'A: all 15'), ('P', 'P: gait + medications')]
sh = {}
for n in ('64', '91'):
    line = re.search(r'n=' + n + r' stroke \d+ (\{.*\})', rob).group(1)
    sh[n] = ast.literal_eval(clean(line))
fig, (P, Q) = plt.subplots(1, 2, figsize=(7.4, 3.5), dpi=250, gridspec_kw={'width_ratios': [1.1, 1]})
for i, (k, lab) in enumerate(sets):
    lo, hi = ci[k]; P.plot([lo, hi], [i, i], color=DK, lw=1.6); P.plot(pt[k], i, 'o', color=DK, ms=6)
    if k in a91: P.plot(a91[k], i, 'D', color=CO, ms=5.5)
P.axvline(0.5, color='grey', lw=0.8); P.set_yticks(range(5)); P.set_yticklabels([s[1] for s in sets], fontsize=8); P.set_ylim(-0.6, 4.6)
P.set_xlim(0.45, 1.02); P.set_xlabel('AUC (refit bootstrap 95% CI)', fontsize=8); P.tick_params(labelsize=8)
P.plot([], [], 'o-', color=DK, label='n = 64'); P.plot([], [], 'D', color=CO, label='n = 91 (point only)')
P.legend(loc='lower right', fontsize=7, frameon=False, bbox_to_anchor=(1, 1), ncol=2, borderaxespad=0.2)
P.text(-0.02, 1.12, '(a)', transform=P.transAxes, fontsize=9, fontweight='bold'); strip(P)
blocks = [('risk', 'Risk factors'), ('gait', 'Gait speed'), ('meds', 'Medications')]
for i, (k, lab) in enumerate(blocks):
    for n, col, d in (('64', DK, -0.17), ('91', CO, 0.17)):
        v, (lo, hi) = sh[n][k]; Q.plot([lo, hi], [i + d, i + d], color=col, lw=1.6); Q.plot(v, i + d, 'o', color=col, ms=6)
Q.axvline(0, color='grey', lw=0.8); Q.set_yticks(range(3)); Q.set_yticklabels([b[1] for b in blocks], fontsize=8); Q.set_ylim(-0.6, 2.6); Q.invert_yaxis()
Q.set_xlabel('Shapley share of AUC above 0.5', fontsize=8); Q.tick_params(labelsize=8)
Q.plot([], [], 'o-', color=DK, label='n = 64'); Q.plot([], [], 'o-', color=CO, label='n = 91')
Q.legend(loc='lower right', fontsize=7, frameon=False, bbox_to_anchor=(1, 1), ncol=2, borderaxespad=0.2)
Q.text(-0.02, 1.12, '(b)', transform=Q.transAxes, fontsize=9, fontweight='bold'); strip(Q)
fig.tight_layout(); fig.savefig(OUT('fig3_clinical.png'), bbox_inches='tight', pad_inches=0.08); plt.show()

## Figure 4. Phase grid (stroke minus control, degrees)

In [ ]:
# ---------- Figure 4: phase grid, read from the 4_grid output ----------
rows = []
pat = re.compile(r'^\s*(\S+)\s+(t60-360|first300|longest300)\s+([LR])\s+(CARNet|surrogate|all bins)\s+(un|pr|ro)\s+(.*)$')
for line in open(GRID, encoding='utf8'):
    m = pat.match(line)
    if m:
        t = list(m.groups()[:5]) + m.group(6).split()
        rows.append(t)   # proto, seg, side, gate, arm, n, S, C, b, lo, hi, p, perm, rob, lev
g = rows
segs=[('t60-360','Tilt, 60 to 360 s'),('first300','Seated, first 300 s'),('longest300','Seated, longest 300 s')]
arms=[('un','unaligned','#7A7A7A'),('pr','per-record','#264653'),('ro','regime-offset','#E76F51')]
gates=[('CARNet','CARNet gate'),('surrogate','Surrogate gate'),('all bins','All bins')]
plt.rcParams['font.family']='DejaVu Sans'
fig,axs=plt.subplots(2,3,figsize=(7.4,6.0),dpi=250,sharex=True,sharey=True)
for ri,side in enumerate(['L','R']):
    for ci,(gk,gl) in enumerate(gates):
        ax=axs[ri][ci]; y=0; ticks=[]; labs=[]
        for sk,sl in segs:
            for ak,al,col in arms:
                r=[x for x in g if x[1]==sk and x[2]==side and x[3]==gk and x[4]==ak]
                assert len(r)==1,(sk,side,gk,ak,len(r))
                r=r[0]; d,lo,hi=float(r[8]),float(r[9]),float(r[10])
                ax.plot([lo,hi],[y,y],color=col,lw=1.5); ax.plot(d,y,'o',color=col,ms=4)
                y+=1
            ticks.append(y-2); labs.append(sl); y+=0.7
        ax.axvline(0,color='k',lw=0.8,ls='--')
        ax.set_title(('Left' if side=='L' else 'Right')+' channel, '+gl,fontsize=8)
        if ci==0: ax.set_yticks(ticks); ax.set_yticklabels(labs,fontsize=7)
        ax.tick_params(axis='x',labelsize=7)
        if ri==1: ax.set_xlabel('Stroke minus control (degrees)',fontsize=7.5)
        for s in ['top','right']: ax.spines[s].set_visible(False)
axs[0][0].invert_yaxis()
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([0],[0],color=c,marker='o',ms=4,lw=1.5,label=l) for _,l,c in arms],loc='lower center',ncol=3,fontsize=8,frameon=False,bbox_to_anchor=(0.5,-0.01))
fig.tight_layout(rect=(0,0.04,1,1)); fig.savefig(OUT('fig4_phase_grid.png'),bbox_inches='tight',pad_inches=0.08); plt.show()